## Execution Guide — Time-Block Baseline CNN (Split v2)

This is the official **Time-Block Baseline CNN** experiment using the frozen
time-block split (v2).

### Modes

Set `MODE` in the RESUME / SETUP cell:

- `"evaluate"` → load the saved checkpoint and evaluate without retraining.
- `"train"` → intentionally retrain the Baseline CNN and save a new checkpoint.

`MODE = "evaluate"` is the safe default.

### Rules

- Use only `data/time_block/`.
- Do not use v0 or v1 checkpoints for official v2 results.
- Do not change the v2 block size or 60-second buffer.
- The test set is used only for final evaluation.
- Models, histories and evaluation artifacts are saved to Google Drive.
- The final `.keras` model stays out of GitHub.
- Run this notebook from top to bottom after a fresh Colab runtime.

# Fresh vs Rotten Fruit Classification — Time-Block Baseline CNN

This notebook trains and evaluates the **Baseline CNN from scratch** on the
official time-block split (v2).

The six classes are:

`freshapples`, `freshbanana`, `freshoranges`,
`rottenapples`, `rottenbanana`, `rottenoranges`.

The split history and leakage investigation are documented in
`02_preprocessing.ipynb`.

## 1. Official Experiment Setup

The official v2 split assigns complete source-photo blocks to train,
validation and test and removes sources that are within the required
60-second cross-split time buffer.

This notebook is only for the Baseline CNN. It does not perform improved-CNN
ablations or MobileNetV2 experiments.

In [ ]:
# =========================================================
# RESUME / SETUP
# Run this cell first after every fresh Colab runtime.
# =========================================================

import json
import os
import sys
import subprocess
from pathlib import Path

MODE = "evaluate"
RUN_SMOKE_TEST = False

REPO_URL = (
    "https://github.com/"
    "onkar-726/fresh-rotten-fruit-classification.git"
)

REPO_DIR = Path(
    "/content/fresh-rotten-fruit-classification"
)

# ---------------------------------------------------------
# Repository
# ---------------------------------------------------------

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Working directory:", Path.cwd())

# ---------------------------------------------------------
# Colab-only packages
# TensorFlow / NumPy are intentionally not pinned here.
# ---------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements-colab.txt",
        "-q",
    ],
    check=True,
)

# ---------------------------------------------------------
# Imports after package installation
# ---------------------------------------------------------

import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

from google.colab import drive

from src.config import (
    CLASS_NAMES,
    NUM_CLASSES,
    IMAGE_SIZE,
    BATCH_SIZE,
    SEED,
    MIN_GAP_SECONDS,
    TRAIN_CSV,
    VALIDATION_CSV,
    TEST_CSV,
)

from src.colab_setup import (
    set_seed,
    print_environment,
    verify_repo,
)

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seed(SEED)

# ---------------------------------------------------------
# Google Drive — mounted once
# ---------------------------------------------------------

drive.mount("/content/drive")

DRIVE_ROOT = (
    Path("/content/drive/MyDrive/fruit_project")
)

DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------
# Verify repository files
# ---------------------------------------------------------

verify_repo(REPO_DIR)

# ---------------------------------------------------------
# Official v2 manifest
# ---------------------------------------------------------

MANIFEST_PATH = (
    REPO_DIR
    / "data"
    / "time_block"
    / "split_manifest.json"
)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing official v2 manifest: {MANIFEST_PATH}"
    )

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    split_manifest = json.load(file)

assert split_manifest["split_version"] == "v2"
assert split_manifest["block_size"] == 25
assert split_manifest["buffer_seconds"] == MIN_GAP_SECONDS
assert (
    split_manifest["closest_cross_split_gap_seconds"]
    >= MIN_GAP_SECONDS
)

# ---------------------------------------------------------
# Dataset download / restore
# ---------------------------------------------------------

dataset_download_path = Path(
    kagglehub.dataset_download(
        "sriramr/fruits-fresh-and-rotten-for-classification"
    )
)

# The KaggleHub location can vary. Find the directory whose
# structure matches the image paths stored in the CSVs.
dataset_candidates = [
    dataset_download_path,
    dataset_download_path / "dataset",
]

if dataset_download_path.exists():
    dataset_candidates.extend(
        p
        for p in dataset_download_path.iterdir()
        if p.is_dir()
    )

DATASET_ROOT = None

for candidate in dataset_candidates:
    if (
        (candidate / "train").is_dir()
        and any((candidate / "train").iterdir())
    ):
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not find the Kaggle dataset root containing the "
        "'train/' image directory."
    )

print("Dataset root:", DATASET_ROOT)

# ---------------------------------------------------------
# Environment
# ---------------------------------------------------------

print_environment()

print("\nTensorFlow version:", tf.__version__)
print("Python version:", sys.version.split()[0])
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Seed:", SEED)
print("Classes:", NUM_CLASSES)
print("MODE:", MODE)
print("RUN_SMOKE_TEST:", RUN_SMOKE_TEST)

print("\nRESUME / SETUP completed successfully.")

## 2. Baseline CNN Architecture

The Baseline CNN architecture is defined centrally in `src/models.py`.

The notebook only verifies the shared model definition and its forward pass.


In [ ]:
# =========================================================
# Architecture check
# =========================================================

from src.models import build_baseline_cnn

architecture_model = build_baseline_cnn()

assert architecture_model.count_params() == 110534

dummy_input = tf.zeros(
    shape=(1, IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    dtype=tf.float32,
)

dummy_output = architecture_model(
    dummy_input,
    training=False,
)

assert dummy_output.shape == (1, NUM_CLASSES)

print("Architecture check: PASSED")
print("Model name:", architecture_model.name)
print("Parameter count:", architecture_model.count_params())
print("Forward-pass output shape:", dummy_output.shape)

## 3. Connect and Verify the Official v2 Dataset

In [ ]:
# =========================================================
# Load official time-block CSVs
# =========================================================

from src.data import load_split

train_df = load_split("train")
validation_df = load_split("validation")
test_df = load_split("test")

split_dfs = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

# ---------------------------------------------------------
# Manifest count checks
# ---------------------------------------------------------

for split_name, df in split_dfs.items():
    expected = split_manifest["image_counts"][split_name]
    assert len(df) == expected, (
        f"{split_name}: expected {expected}, got {len(df)}"
    )

# ---------------------------------------------------------
# All classes present
# ---------------------------------------------------------

for split_name, df in split_dfs.items():
    assert set(df["class_name"]) == set(CLASS_NAMES), (
        f"Missing class in {split_name}"
    )

# ---------------------------------------------------------
# Zero source overlap
# ---------------------------------------------------------

source_sets = {
    name: set(df["source_id"].unique())
    for name, df in split_dfs.items()
}

assert source_sets["train"].isdisjoint(
    source_sets["validation"]
)
assert source_sets["train"].isdisjoint(
    source_sets["test"]
)
assert source_sets["validation"].isdisjoint(
    source_sets["test"]
)

# ---------------------------------------------------------
# Independent minimum capture-time gap check
# ---------------------------------------------------------

source_records = (
    pd.concat(
        [
            df[
                [
                    "source_id",
                    "class_name",
                    "capture_time",
                ]
            ]
            for df in split_dfs.values()
        ],
        ignore_index=True,
    )
    .drop_duplicates("source_id")
)

source_records["capture_time"] = pd.to_datetime(
    source_records["capture_time"]
)

source_records["split"] = (
    source_records["source_id"].map(
        {
            source_id: split_name
            for split_name, ids in source_sets.items()
            for source_id in ids
        }
    )
)

minimum_gap = float("inf")

for _, group in source_records.groupby("class_name"):
    times = group["capture_time"].to_numpy()
    splits = group["split"].to_numpy()

    gaps = (
        np.abs(
            times[:, None] - times[None, :]
        )
        / np.timedelta64(1, "s")
    )

    different = (
        splits[:, None] != splits[None, :]
    )

    if different.any():
        minimum_gap = min(
            minimum_gap,
            float(gaps[different].min()),
        )

assert minimum_gap >= MIN_GAP_SECONDS

# ---------------------------------------------------------
# Blocks per class
# ---------------------------------------------------------

for split_name, df in split_dfs.items():
    blocks = (
        df.groupby("class_name")["block_id"]
        .nunique()
        .to_dict()
    )

    print(f"\n{split_name} blocks per class:")
    print(blocks)

print("\nOfficial split verification: PASSED")
print(
    "Train:",
    len(train_df),
    "images /",
    train_df["source_id"].nunique(),
    "sources",
)
print(
    "Validation:",
    len(validation_df),
    "images /",
    validation_df["source_id"].nunique(),
    "sources",
)
print(
    "Test:",
    len(test_df),
    "images /",
    test_df["source_id"].nunique(),
    "sources",
)
print(
    "Minimum cross-split capture-time gap:",
    f"{minimum_gap:.0f} seconds",
)

## 4. Create TensorFlow Datasets and Run the Smoke Test

In [ ]:
# =========================================================
# TensorFlow datasets
# =========================================================

from src.data import build_dataset

train_ds = build_dataset(
    train_df,
    dataset_root=DATASET_ROOT,
    training=True,
    augment=False,
    mobilenet=False,
)

validation_ds = build_dataset(
    validation_df,
    dataset_root=DATASET_ROOT,
    training=False,
    augment=False,
    mobilenet=False,
)

test_ds = build_dataset(
    test_df,
    dataset_root=DATASET_ROOT,
    training=False,
    augment=False,
    mobilenet=False,
)

# ---------------------------------------------------------
# Dataset sanity check
# ---------------------------------------------------------

for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

    assert images.shape[1:] == (
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3,
    )

    assert labels.shape[0] <= BATCH_SIZE
    assert float(tf.reduce_min(images)) >= 0.0
    assert float(tf.reduce_max(images)) <= 1.0

print("Dataset pipeline check: PASSED")

# ---------------------------------------------------------
# Optional training smoke test
# ---------------------------------------------------------

if RUN_SMOKE_TEST:

    print("\nRUN_SMOKE_TEST=True")

    smoke_model = build_baseline_cnn()

    smoke_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    smoke_history = smoke_model.fit(
        train_ds.take(2),
        validation_data=validation_ds.take(1),
        epochs=1,
        verbose=1,
    )

    print(
        "Smoke-test accuracy:",
        smoke_history.history["accuracy"][-1],
    )

    print("Training smoke test: PASSED")
else:
    print("\nRUN_SMOKE_TEST=False — full training smoke test skipped.")

## 5. Train the Time-Block Baseline CNN

The Baseline CNN intentionally uses the original baseline training recipe:

- no training augmentation
- no class weights
- Adam with learning rate `1e-3`
- EarlyStopping on validation loss
- ReduceLROnPlateau
- best checkpoint selected using validation loss

The test set is not used during training.

In [ ]:
# =========================================================
# Baseline CNN training
# =========================================================

import json

from src.evaluate import plot_history

MODEL_DIR = DRIVE_ROOT / "models"

RESULTS_DIR = (
    DRIVE_ROOT
    / "results"
    / "baseline_cnn_time_block"
)

HISTORY_DIR = RESULTS_DIR / "history"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HISTORY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_NAME = (
    "baseline_cnn_time_block_best.keras"
)

MODEL_PATH = MODEL_DIR / MODEL_NAME

HISTORY_CSV_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_history.csv"
)

HISTORY_JSON_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_history.json"
)

EPOCHS = 15
LEARNING_RATE = 1e-3

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

if MODE == "train":

    tf.keras.backend.clear_session()

    model = build_baseline_cnn()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print("Training Baseline CNN...")
    print("Parameters:", model.count_params())
    print("Checkpoint:", MODEL_PATH)

    history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    history_data = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }

    with open(
        HISTORY_JSON_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            history_data,
            file,
            indent=2,
        )

    plot_history(
        history,
        output_dir=HISTORY_DIR,
        prefix="baseline_cnn_time_block",
    )

    best_epoch = (
        min(
            range(len(history.history["val_loss"])),
            key=lambda i: history.history["val_loss"][i],
        )
        + 1
    )

    print("\nTraining completed.")
    print("Epochs:", len(history.history["loss"]))
    print("Best epoch:", best_epoch)
    print(
        "Best validation loss:",
        history.history["val_loss"][best_epoch - 1],
    )
    print(
        "Best validation accuracy:",
        history.history["val_accuracy"][best_epoch - 1],
    )

    assert MODEL_PATH.exists()
    assert HISTORY_CSV_PATH.exists()
    assert HISTORY_JSON_PATH.exists()

elif MODE == "evaluate":

    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Saved Baseline CNN checkpoint not found:\n"
            f"{MODEL_PATH}\n\n"
            "Set MODE='train' to create a new v2 checkpoint."
        )

    if not HISTORY_JSON_PATH.exists():
        raise FileNotFoundError(
            f"Saved training history not found:\n"
            f"{HISTORY_JSON_PATH}"
        )

    with open(
        HISTORY_JSON_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        history_data = json.load(file)

    best_epoch = (
        min(
            range(len(history_data["val_loss"])),
            key=lambda i: history_data["val_loss"][i],
        )
        + 1
    )

    print("MODE='evaluate' — no training performed.")
    print("Saved history loaded.")
    print("Recorded epochs:", len(history_data["loss"]))
    print("Best epoch:", best_epoch)
    print(
        "Best validation loss:",
        history_data["val_loss"][best_epoch - 1],
    )
    print(
        "Best validation accuracy:",
        history_data["val_accuracy"][best_epoch - 1],
    )

else:

    raise ValueError(
        "MODE must be 'train' or 'evaluate'."
    )

## 6. Load the Saved Time-Block Baseline CNN

Always evaluate the uniquely named v2 checkpoint.

The upload option exists for recovery after a runtime reset; the filename is
checked so a checkpoint from another experiment cannot be loaded silently.

In [ ]:
# =========================================================
# Load the saved Baseline CNN
# =========================================================

from pathlib import Path

EXPECTED_MODEL_NAME = (
    "baseline_cnn_time_block_best.keras"
)

LOAD_METHOD = "drive"

if LOAD_METHOD == "drive":

    MODEL_PATH = (
        DRIVE_ROOT
        / "models"
        / EXPECTED_MODEL_NAME
    )

    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Expected model not found: {MODEL_PATH}"
        )

elif LOAD_METHOD == "upload":

    from google.colab import files

    uploaded = files.upload()

    assert len(uploaded) == 1, (
        "Upload exactly one .keras model file."
    )

    uploaded_name = next(iter(uploaded))

    assert uploaded_name == EXPECTED_MODEL_NAME, (
        f"Wrong model file: {uploaded_name}. "
        f"Expected: {EXPECTED_MODEL_NAME}"
    )

    MODEL_PATH = (
        Path("/content")
        / uploaded_name
    )

else:

    raise ValueError(
        "LOAD_METHOD must be 'drive' or 'upload'."
    )

assert MODEL_PATH.name == EXPECTED_MODEL_NAME
assert MODEL_PATH.exists()

print("Loading:", MODEL_PATH)

model = tf.keras.models.load_model(
    MODEL_PATH
)

assert model.count_params() == 110534

print("\nModel loaded successfully.")
print("Model name:", model.name)
print("Parameters:", model.count_params())

## 7. Full Time-Block Test Evaluation

This is the final evaluation on the complete official v2 test set in the official v2
test split.

In [ ]:
# =========================================================
# Full test evaluation and export
# =========================================================

from src.evaluate import (
    evaluate_model,
    export_results,
)

FULL_RESULTS_DIR = (
    RESULTS_DIR
    / "full_test"
)

FULL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1,
)

y_true, y_pred, y_prob = evaluate_model(
    model,
    test_ds,
)

assert len(y_true) == len(test_df)
assert len(y_pred) == len(test_df)
assert y_prob.shape[0] == len(test_df)

full_summary = export_results(
    y_true=y_true,
    y_pred=y_pred,
    probabilities=y_prob,
    class_names=CLASS_NAMES,
    output_dir=FULL_RESULTS_DIR,
    prefix="baseline_cnn_time_block_full",
    metadata={
        "split": "time_block_v2",
        "test_view": "full",
        "test_images": len(test_df),
        "test_sources": int(
            test_df["source_id"].nunique()
        ),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "parameters": int(model.count_params()),
        "tensorflow_version": tf.__version__,
        "seed": SEED,
    },
)

# Add source/class metadata to the prediction CSV.
full_predictions = test_df[
    [
        "image_path",
        "class_name",
        "source_filename",
        "source_id",
        "capture_time",
        "block_id",
    ]
].reset_index(drop=True).copy()

full_predictions["true_label"] = y_true
full_predictions["predicted_label"] = y_pred
full_predictions["true_class"] = [
    CLASS_NAMES[i] for i in y_true
]
full_predictions["predicted_class"] = [
    CLASS_NAMES[i] for i in y_pred
]
full_predictions["confidence"] = np.max(
    y_prob,
    axis=1,
)

full_predictions_path = (
    FULL_RESULTS_DIR
    / "baseline_cnn_time_block_full_predictions.csv"
)

full_predictions.to_csv(
    full_predictions_path,
    index=False,
)

print("\nFull test results")
print("Loss:", test_loss)
print("Accuracy:", test_accuracy)
print(
    "Macro F1:",
    full_summary["macro_f1"],
)
print(
    "Weighted F1:",
    full_summary["weighted_f1"],
)

print(
    "\nFull-test artifacts saved to:",
    FULL_RESULTS_DIR,
)

## 8. Original-Only Test Evaluation

The original-only test view keeps exactly one non-augmented image for each source photo
photo. It provides a second test view without changing the official split.

In [ ]:
# =========================================================
# Original-only test evaluation and export
# =========================================================

from src.data import original_only

ORIGINAL_RESULTS_DIR = (
    RESULTS_DIR
    / "original_only"
)

ORIGINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

original_test_df = original_only(
    test_df
)

assert (
    len(original_test_df)
    == original_test_df["source_id"].nunique()
)

assert set(
    original_test_df["class_name"]
) == set(CLASS_NAMES)

original_test_ds = build_dataset(
    original_test_df,
    dataset_root=DATASET_ROOT,
    training=False,
    augment=False,
    mobilenet=False,
)

original_test_loss, original_test_accuracy = (
    model.evaluate(
        original_test_ds,
        verbose=1,
    )
)

original_y_true, original_y_pred, original_y_prob = (
    evaluate_model(
        model,
        original_test_ds,
    )
)

assert len(original_y_true) == len(
    original_test_df
)

original_summary = export_results(
    y_true=original_y_true,
    y_pred=original_y_pred,
    probabilities=original_y_prob,
    class_names=CLASS_NAMES,
    output_dir=ORIGINAL_RESULTS_DIR,
    prefix="baseline_cnn_time_block_original_only",
    metadata={
        "split": "time_block_v2",
        "test_view": "original_only",
        "test_images": len(original_test_df),
        "test_sources": int(
            original_test_df["source_id"].nunique()
        ),
        "test_loss": float(original_test_loss),
        "test_accuracy": float(original_test_accuracy),
        "parameters": int(model.count_params()),
        "tensorflow_version": tf.__version__,
        "seed": SEED,
    },
)

original_predictions = original_test_df[
    [
        "image_path",
        "class_name",
        "source_filename",
        "source_id",
        "capture_time",
        "block_id",
    ]
].reset_index(drop=True).copy()

original_predictions["true_label"] = (
    original_y_true
)
original_predictions["predicted_label"] = (
    original_y_pred
)
original_predictions["true_class"] = [
    CLASS_NAMES[i] for i in original_y_true
]
original_predictions["predicted_class"] = [
    CLASS_NAMES[i] for i in original_y_pred
]
original_predictions["confidence"] = np.max(
    original_y_prob,
    axis=1,
)

original_predictions_path = (
    ORIGINAL_RESULTS_DIR
    / "baseline_cnn_time_block_original_only_predictions.csv"
)

original_predictions.to_csv(
    original_predictions_path,
    index=False,
)

print("\nOriginal-only results")
print("Images:", len(original_test_df))
print("Loss:", original_test_loss)
print("Accuracy:", original_test_accuracy)
print(
    "Macro F1:",
    original_summary["macro_f1"],
)
print(
    "Weighted F1:",
    original_summary["weighted_f1"],
)

print(
    "\nOriginal-only artifacts saved to:",
    ORIGINAL_RESULTS_DIR,
)

## 9. Top 12 Misclassified Test Images

The following section shows up to 12 incorrect predictions from the official
v2 test set, ranked by prediction confidence, with true class, predicted class
and confidence.

In [ ]:
# =========================================================
# Top 12 misclassified images
# =========================================================

import matplotlib.pyplot as plt
from IPython.display import display

misclassified_mask = (
    y_true != y_pred
)

misclassified = full_predictions.loc[
    misclassified_mask,
    [
        "image_path",
        "true_class",
        "predicted_class",
        "confidence",
        "source_id",
    ]
].copy()

misclassified = (
    misclassified
    .sort_values("confidence", ascending=False)
    .head(12)
    .reset_index(drop=True)
)

MISCLASSIFIED_DIR = (
    RESULTS_DIR
    / "misclassified_test"
)

MISCLASSIFIED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

misclassified_csv = (
    MISCLASSIFIED_DIR
    / "baseline_cnn_time_block_top12_misclassified.csv"
)

misclassified.to_csv(
    misclassified_csv,
    index=False,
)

print(
    "Misclassified test images:",
    int(misclassified_mask.sum()),
)

print("\nTop misclassified samples:")
display(misclassified)

n_show = min(12, len(misclassified))

if n_show > 0:

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(14, 10),
    )

    axes = axes.ravel()

    for ax in axes:
        ax.axis("off")

    for i in range(n_show):

        row = misclassified.iloc[i]

        image_path = (
            DATASET_ROOT
            / row["image_path"]
        )

        if image_path.exists():

            image = plt.imread(
                str(image_path)
            )

            axes[i].imshow(image)

            axes[i].set_title(
                f"True: {row['true_class']}\n"
                f"Pred: {row['predicted_class']}\n"
                f"Conf: {row['confidence']:.3f}"
            )

        else:

            axes[i].text(
                0.5,
                0.5,
                "Image not found",
                ha="center",
                va="center",
            )

        axes[i].axis("off")

    plt.tight_layout()

    misclassified_png = (
        MISCLASSIFIED_DIR
        / "baseline_cnn_time_block_top12_misclassified.png"
    )

    plt.savefig(
        misclassified_png,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Saved:",
        misclassified_png,
    )

else:

    print("No misclassified test images were found.")

## 10. Save Experiment Metadata and Copy Small Results to the Repository

The trained `.keras` checkpoint remains in Google Drive.

Only small, verifiable result artifacts are copied into the Git repository.

In [ ]:
# =========================================================
# Final metadata + repository result export
# =========================================================

import shutil
from pathlib import Path

experiment_metadata = {
    "experiment": "baseline_cnn_time_block",
    "split": "time_block_v2",
    "model_checkpoint": MODEL_NAME,
    "parameters": int(model.count_params()),
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "learning_rate": LEARNING_RATE,
    "epochs_limit": EPOCHS,
    "training_augmentation": False,
    "class_weights": False,
    "test_images": int(len(test_df)),
    "test_sources": int(
        test_df["source_id"].nunique()
    ),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(
        full_summary["macro_f1"]
    ),
    "test_weighted_f1": float(
        full_summary["weighted_f1"]
    ),
    "original_only_images": int(
        len(original_test_df)
    ),
    "original_only_accuracy": float(
        original_test_accuracy
    ),
    "original_only_macro_f1": float(
        original_summary["macro_f1"]
    ),
    "tensorflow_version": tf.__version__,
}

metadata_path = (
    RESULTS_DIR
    / "baseline_cnn_time_block_experiment_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------
# Copy small result files to GitHub repo
# ---------------------------------------------------------

REPO_RESULTS_DIR = (
    REPO_DIR
    / "results"
    / "time_block"
    / "baseline_cnn"
)

REPO_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

source_files = []

for folder in [
    HISTORY_DIR,
    FULL_RESULTS_DIR,
    ORIGINAL_RESULTS_DIR,
    MISCLASSIFIED_DIR,
]:

    if folder.exists():
        source_files.extend(
            path
            for path in folder.rglob("*")
            if path.is_file()
        )

source_files.append(metadata_path)

copied = []

for source in source_files:

    destination = (
        REPO_RESULTS_DIR
        / source.name
    )

    shutil.copy2(
        source,
        destination,
    )

    copied.append(destination)

print("Copied result files:")
for destination in sorted(copied):
    print(" -", destination)

print(
    "\nModel checkpoint remains in Drive:",
    MODEL_PATH,
)

print(
    "\nFinal Baseline CNN result export completed."
)

## 11. Final Status

This notebook is the cleaned reference workflow for the official Time-Block
Baseline CNN.

The notebook is complete when:

- official v2 split verification passes,
- the saved or newly trained v2 checkpoint is loaded,
- full-test results are exported,
- original-only results are exported,
- top misclassified test images are saved,
- histories/plots are saved to Drive,
- small result artifacts are copied to `results/time_block/baseline_cnn/`.

Do not type performance numbers manually into this notebook or README.
Use the saved JSON/CSV reports generated above as the source of truth.